In [ ]:
import os

from dotenv import load_dotenv
import folium
import polyline
import requests
import urllib3

HTTP_PROXY = 'http://sia-lb.telekom.de:8080'
HTTPS_PROXY = 'http://sia-lb.telekom.de:8080'

os.environ['http_proxy'] = HTTP_PROXY
os.environ['https_proxy'] = HTTPS_PROXY

In [ ]:
# .env Datei laden (optional mit Pfad: load_dotenv(dotenv_path="/pfad/zur/.env"))
load_dotenv()

# Zugriff auf Umgebungsvariablen
CLIENT_ID = os.getenv('CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET')
REFRESH_TOKEN = os.getenv('REFRESH_TOKEN')
GRANT_TYPE = os.getenv('GRANT_TYPE')

In [ ]:
"""
This script is for getting activity information from Strava.
Use Case: Data Analysis
Author: Xaver Heuser (xaver.heuser@gmail.com)
Date: 15.10.2024

Link to Stra API: https://developers.strava.com/docs/reference/

"""


urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


################################
# Read Config File
################################

# with open('config.json', 'r') as f:
#     config = json.load(f)

activities_url = 'https://www.strava.com/api/v3/athlete/activities'
auth_url = 'https://www.strava.com/oauth/token'


################################
# Authentication
################################

# payload = {
#     'client_id': config['client_id'],
#     'client_secret': config['client_secret'],
#     'refresh_token': config['refresh_token'],
#     'grant_type': config['grant_type'],
#     'f': 'json'
# }
payload = {
    'client_id': CLIENT_ID,
    'client_secret': CLIENT_SECRET,
    'refresh_token': REFRESH_TOKEN,
    'grant_type': GRANT_TYPE,
    'f': 'json',
}

print('Requesting Token...\n')
res = requests.post(auth_url, data=payload, verify=False)
access_token = res.json()['access_token']
print(f'Access Token = {access_token} \n')

header = {'Authorization': 'Bearer ' + access_token}


################################
# Request Activities
################################

# The first loop, request_page_number will be set to one, so it requests the first page. Increment this number after
# each request, so the next time we request the second page, then third, and so on...
request_page_num = 1
all_activities = []

while True:
    param = {'per_page': 200, 'page': request_page_num}
    # initial request, where we request the first page of activities
    my_dataset = requests.get(activities_url, headers=header, params=param).json()

    # check the response to make sure it is not empty. If it is empty, that means there is no more data left. So if you have
    # 1000 activities, on the 6th request, where we request page 6, there would be no more data left, so we will break out of the loop
    if len(my_dataset) == 0:
        break

    # if the all_activities list is already populated, that means we want to add additional data to it via extend.
    if all_activities:
        all_activities.extend(my_dataset)

    # if the all_activities is empty, this is the first time adding data so we just set it equal to my_dataset
    else:
        all_activities = my_dataset

    request_page_num += 1

print('The length of all activies is: ', len(all_activities))
for activity in all_activities:
    print(activity)
    print(activity['name'])
    print(activity['distance'] / 1000, 'km')
    print(activity['elapsed_time'] / 60, 'min')
    print(10 * '=')

In [ ]:
# Deine Polyline-Zeichenkette
polyline_string = 'waluH_q|j@rN`EnCbJoi@n}AqPjjA{L`g@cGfBkJqKeE}@uTvOiB`J~B`Uk@|KuKvSeLxb@uhA~kB{CrIm@dI`AtM|Yl_Bsh@`Oc`@Pq[|Jeb@b@eLsF{Zgd@aN{FmLhc@iBjWi}@j\\{j@woAyX_f@cq@cvAiNwXsAuIt@cf@tD{Lx@wKsBg\\cFwXgCuDaI{B}IwJaSqd@{VmQxW_VjEgZzEabBc@_JuEoP`CqJjBiVbKsDfNoMrBmRpE_NtHyc@jPw[lXwm@zDuOvWkB~CaFfDeh@}Lea@nCgNNeScF_PmAmU`EoQaFw{@v@m^lAwLfCkH`PyMvJkQfCkSgFsB}H~CsIuQeJ|A_B~FuA|TeWr_@uFrPyNfNiDhAuOqMkWkHyKfFkGbMoKjA_DiC_IkQcU}iAuLiX[uLeI_NcDaOuPi]oJeP{H_DkEsFyCsG_DeMe@gDeJ_FtIxErCrFbChNzJxMcK{NoBkMcB}@q@uDaO{KuF~L_C]_EwL`@oGwDyBkH}_@_GcIaDlGcDwGyOwo@oNkX{BeKiPs]_CmK_NsOwHeYcNoLcIl@{G{BiLeM}IoUs]q_Bk@uLAkD|AbAf@fEhEzGdBM~@uDx@wT_AgIpCuMl@oWdFwLm@qN~AyBx@gPkI}V_Fw[wEaRzKwG`MlBlLiAdDeJpFhAbBsH|K|@fHoFx^~ElKr_@nMtJnD_AnO}NhR_BuFrc@|@l]j`AlnCdTxX`E~SzEgG~KRnJvKzHX|CxHlKxC`D[jD{FbMuBjEpEbSoDxIjJ~ImMxViMnKbC~DeDhEzHjG}BpDdHxE~AxF`ItJx`@xMas@fIuNpOmp@~@_QvM}YdIvAjXr[pOaChM_Q`N}CrJkOxZ{Tr\\_q@nBsRdJ`LaDmGnEmb@}B}^x@{HdIvMrt@tJvWrTrK`D~ExEzFdOfF`f@u@zn@lGbR|C~b@sCpVFpQtFn[r@xQjHbP]bVtB~NzFbDvQ}MdH`B|Sd\\|DhZhSlRdKvEbDrP|IxF|ElIrApEhAhVh]b`@nAbFzApU~r@wA~ElDg@pu@gMzc@_E|l@NxKjDbQA|JuJ~j@sCfl@~M|o@jDtHlKxKpCxKI|K_G~Xx@|VyEnVhFlVvHd|@cFnb@wGfOuF|EuXnKUtBeT~QyLtReQdA}IdEyDrGsKx^{bAboAuZrTsX`KcKhLsThGgb@nX{BxFbUfKdMcCjS|StKjPlF}IzNdGdXk@pNlKtEInVeOzFsLlC{OxI{O`Lp@tXeJnCiFyAcThDsPx@wS'

# 1. Polyline-String in eine Liste von Koordinaten (Latitude, Longitude) dekodieren
decoded_coords = polyline.decode(polyline_string)

# 2. Den Startpunkt der Route finden, um die Karte zu zentrieren
if decoded_coords:
    center_point = decoded_coords[0]
else:
    print('Die Polyline ist leer. Es können keine Koordinaten dekodiert werden.')

# 3. Eine Folium-Karte erstellen
# Der Parameter 'zoom_start' bestimmt den anfänglichen Zoom-Level der Karte
m = folium.Map(location=center_point, zoom_start=13)

# 4. Die Polyline zur Karte hinzufügen
folium.PolyLine(locations=decoded_coords, color='blue', weight=5, opacity=0.8).add_to(m)

# 5. Optional: Start- und Endpunkt der Route markieren
folium.Marker(
    location=decoded_coords[0], popup='Startpunkt', icon=folium.Icon(color='green')
).add_to(m)
folium.Marker(
    location=decoded_coords[-1], popup='Endpunkt', icon=folium.Icon(color='red')
).add_to(m)

# 6. Die Karte in der Notebook-Zelle anzeigen
m